# Sentiment Scoring Only (Qwen fine-tuned adapter)

Runs the expensive/GPU-bound part of the pipeline exactly once: streams FNSPID from
Hugging Face (no full download), classifies every matched headline with the fine-tuned
Qwen2.5-3B-Instruct + LoRA adapter, and saves the headline-level scores to `scored_news.csv`.

This notebook does **not** touch stock prices, technical indicators, or RL training --
those stay local (fast, no GPU needed). Download `scored_news.csv` from Kaggle's Output
tab when this finishes and bring it back for the merge + training step.


### Install packages

In [10]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.1 MB/s  0:00:00


In [1]:
!pip install --upgrade pip -q

# HuggingFace stack (Kaggle provides CUDA already)
!pip install "transformers>=4.46" tokenizers huggingface-hub datasets peft accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.3 MB/s eta 0:00:0000:010:01


In [2]:
# ── Sample-mode toggle ──────────────────────────────────────────────
# Streaming all of FNSPID + classifying ~60k headlines with Qwen takes 30+ minutes, and a
# real run already hit two failures deep into that (a KeyError near row 28.5M, and a
# missing HF_TOKEN secret) -- expensive to discover that late. Run once with
# SAMPLE_MODE = True first to smoke-test the whole notebook end-to-end (streaming, HF
# login, model loading, classify_headlines, CSV writing) in a couple of minutes. Once that
# runs clean, flip this to False and use Kernel > Restart & Run All for the real run.
SAMPLE_MODE = False
SAMPLE_MAX_STREAMED_ROWS = 300_000       # stop streaming FNSPID after this many raw rows
SAMPLE_MAX_HEADLINES_TO_CLASSIFY = 200   # cap on how many matched headlines Qwen scores


### Stream FNSPID (no full dataset download -- only matching rows kept)

In [4]:
from datasets import load_dataset
import pandas as pd

TOP10 = set(['AAPL','MSFT','AMZN','GOOGL','META',
             'NVDA','JPM','TSLA','BA','DIS'])
DATE_MIN = pd.Timestamp("2020-01-01")
DATE_MAX = pd.Timestamp("2023-12-31 23:59:59")

# FNSPID has a single "train" split -- no train/test split on the HF side.
ds = load_dataset("Zihan1004/FNSPID", split="train", streaming=True)
# Drop the heavy text/summary columns we don't use -- cuts bandwidth substantially since the
# underlying parquet is columnar. This dataset's repo metadata is incomplete, so
# ds.column_names can come back None instead of a list -- fall back to a no-op in that case
# (correctness doesn't depend on this, it's purely a bandwidth optimization).
_known_cols = ds.column_names or []
_drop_cols = [c for c in ["Url", "Publisher", "Author", "Article",
                           "Lsa_summary", "Luhn_summary",
                           "Textrank_summary", "Lexrank_summary"]
              if c in _known_cols]
if _drop_cols:
    ds = ds.remove_columns(_drop_cols)

keep_rows = []
batch = {"Date": [], "Article_title": [], "Stock_symbol": []}
BATCH_SIZE = 10_000
n_seen = 0
n_skipped_malformed = 0

def _flush(batch, keep_rows):
    if not batch["Date"]:
        return
    chunk = pd.DataFrame(batch)
    chunk["Date_parsed"] = pd.to_datetime(chunk["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    mask = (chunk["Date_parsed"] >= DATE_MIN) & (chunk["Date_parsed"] <= DATE_MAX) & chunk["Stock_symbol"].isin(TOP10)
    sub = chunk.loc[mask, ["Date", "Article_title", "Stock_symbol"]]
    if not sub.empty:
        keep_rows.append(sub)

for row in ds:
    n_seen += 1
    # FNSPID's schema isn't fully uniform across every shard -- a real run hit a row
    # missing the "Date" key entirely around row 28.5M and crashed. Skip malformed rows
    # instead of crashing; they can't be matched to a ticker/date anyway.
    date, title, symbol = row.get("Date"), row.get("Article_title"), row.get("Stock_symbol")
    if date is None or title is None or symbol is None:
        n_skipped_malformed += 1
    else:
        batch["Date"].append(date)
        batch["Article_title"].append(title)
        batch["Stock_symbol"].append(symbol)

    if len(batch["Date"]) >= BATCH_SIZE:
        _flush(batch, keep_rows)
        batch = {"Date": [], "Article_title": [], "Stock_symbol": []}

    if n_seen % 500_000 == 0:
        kept_so_far = sum(len(k) for k in keep_rows)
        print(f"streamed {n_seen:,} rows so far, kept {kept_so_far:,} "
              f"(skipped {n_skipped_malformed:,} malformed)")

    if SAMPLE_MODE and n_seen >= SAMPLE_MAX_STREAMED_ROWS:
        print(f"[SAMPLE_MODE] stopping early after {n_seen:,} streamed rows")
        break

_flush(batch, keep_rows)  # last partial batch

if keep_rows:
    df_news = pd.concat(keep_rows, ignore_index=True)
else:
    df_news = pd.DataFrame(columns=["Date", "Article_title", "Stock_symbol"])

df_news = df_news.rename(columns={
    "Date": "date",
    "Article_title": "headline",
    "Stock_symbol": "ticker",
})
df_news["date"] = pd.to_datetime(df_news["date"], utc=True).dt.tz_localize(None)

print("Final shape:", df_news.shape, "| skipped malformed rows:", n_skipped_malformed)
df_news.head()


NameError: name 'Zihan1004' is not defined

In [5]:
# Recovers df_news from an interrupted streaming cell using whatever was already
# flushed into keep_rows, instead of re-streaming from scratch.
if batch["Date"]:
    _flush(batch, keep_rows)  # catch any not-yet-flushed partial batch

df_news = pd.concat(keep_rows, ignore_index=True) if keep_rows else pd.DataFrame(columns=["Date", "Article_title", "Stock_symbol"])
df_news = df_news.rename(columns={"Date": "date", "Article_title": "headline", "Stock_symbol": "ticker"})
df_news["date"] = pd.to_datetime(df_news["date"], utc=True).dt.tz_localize(None)

print(f"Recovered {df_news.shape[0]} matched headlines from interrupted stream (n_seen={n_seen:,})")
df_news.head()


Recovered 60343 matched headlines from interrupted stream (n_seen=53,025,678)


,date,headline,ticker
0,2020-06-10 07:33:26,Tech Stocks And FAANGS Strong Again To Start D...,AAPL
1,2020-06-10 04:14:08,10 Biggest Price Target Changes For Wednesday,AAPL
2,2020-06-10 03:53:47,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL
3,2020-06-10 03:19:25,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL
4,2020-06-10 02:27:11,Apple To Let Users Trade In Their Mac Computer...,AAPL


In [6]:
news_out = "news_SAMPLE.csv" if SAMPLE_MODE else "news.csv"
df_news.to_csv(news_out, index=False)
print(f"Saved {len(df_news)} ticker-matched headlines to {news_out} "
      "(checkpoint -- if the Qwen step below fails, reload this instead of re-streaming: "
      f'df_news = pd.read_csv("{news_out}", parse_dates=["date"]))')


Saved 60343 ticker-matched headlines to news.csv (checkpoint -- if the Qwen step below fails, reload this instead of re-streaming: df_news = pd.read_csv("news.csv", parse_dates=["date"]))


### Load fine-tuned Qwen adapter and classify every headline

**Before running this cell**, the adapter repo is private, so Kaggle needs your HF token
as a secret attached to *this* kernel:
1. In the notebook editor, open **Add-ons > Secrets**.
2. **Add a new secret** with label exactly `HF_TOKEN` (case-sensitive) and your token as
   the value (starts with `hf_...`).
3. **Toggle it on/attached for this specific notebook** — adding a secret to your account
   does not automatically attach it to every notebook; there's a per-notebook checkbox in
   the Secrets panel that must be enabled here too.
4. Re-run this cell (no restart needed).

If you see `No user secrets exist for kernel id ... and label HF_TOKEN`, it means step 2
or step 3 above wasn't done for this kernel.


In [11]:
# Add your HF token as a Kaggle secret named HF_TOKEN (Add-ons > Secrets), or set the
# HF_TOKEN environment variable before running -- needed since the adapter repo is private.
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login
import os

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"  # your fine-tuned adapter (private repo)

if "HF_TOKEN" in os.environ:
    login(os.environ["HF_TOKEN"])
else:
    try:
        from kaggle_secrets import UserSecretsClient
        login(UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception as e:
        raise RuntimeError(
            "Could not find an HF token. On Kaggle: Add-ons > Secrets > add HF_TOKEN, "
            "enable it for this notebook, then re-run this cell."
        ) from e

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"  # last token of every padded sequence is always the true final token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.to(device)
model.eval()

# Each label is verified to be a single token for this tokenizer.
LABEL_IDS = {
    "positive": tokenizer.encode("positive", add_special_tokens=False)[0],
    "negative": tokenizer.encode("negative", add_special_tokens=False)[0],
    "neutral":  tokenizer.encode("neutral",  add_special_tokens=False)[0],
}
LABEL_ID_LIST = [LABEL_IDS["positive"], LABEL_IDS["negative"], LABEL_IDS["neutral"]]

TICKER_TO_COMPANY = {
    "AAPL": "Apple", "MSFT": "Microsoft", "AMZN": "Amazon", "GOOGL": "Alphabet",
    "META": "Meta", "NVDA": "Nvidia", "JPM": "JPMorgan Chase", "TSLA": "Tesla",
    "BA": "Boeing", "DIS": "Disney",
}

SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def make_prompt(headline, ticker):
    company = TICKER_TO_COMPANY.get(ticker, ticker)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nHeadline: {headline}\nCompany: {company}\nSentiment:<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

@torch.no_grad()
def classify_headlines(texts, tickers, batch_size=32, log_every=500):
    """Returns a continuous score per headline: P(positive) - P(negative), in [-1, 1] --
    read directly from the model's own next-token probability distribution
    (a single forward pass, no generation)."""
    results, t0 = [], time.time()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_tickers = tickers[i:i + batch_size]
        prompts = [make_prompt(h, t) for h, t in zip(batch_texts, batch_tickers)]
        enc = tokenizer(prompts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        logits = model(**enc).logits[:, -1, :]
        label_logits = logits[:, LABEL_ID_LIST]
        probs = torch.softmax(label_logits, dim=-1)
        scores = (probs[:, 0] - probs[:, 1]).float().cpu().tolist()
        results.extend(scores)
        if i % log_every == 0 and i > 0:
            elapsed = time.time() - t0
            rate = i / elapsed
            eta = (len(texts) - i) / rate
            print(f"{i}/{len(texts)} processed ({elapsed:.1f}s elapsed, ~{eta:.0f}s remaining)")
    return results


Using device: cuda


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


adapter_model.safetensors:   0%|          | 0.00/240M [00:00<?, ?B/s]

In [12]:
if SAMPLE_MODE:
    df_news = df_news.head(SAMPLE_MAX_HEADLINES_TO_CLASSIFY).copy()
    print(f"[SAMPLE_MODE] classifying only {len(df_news)} matched headlines")

df_news["score"] = classify_headlines(df_news["headline"].tolist(), df_news["ticker"].tolist())
df_news.head()


4000/60343 processed (211.0s elapsed, ~2972s remaining)
8000/60343 processed (421.8s elapsed, ~2760s remaining)
12000/60343 processed (637.8s elapsed, ~2570s remaining)
16000/60343 processed (847.1s elapsed, ~2348s remaining)
20000/60343 processed (1047.0s elapsed, ~2112s remaining)
24000/60343 processed (1237.1s elapsed, ~1873s remaining)
28000/60343 processed (1424.3s elapsed, ~1645s remaining)
32000/60343 processed (1618.2s elapsed, ~1433s remaining)
36000/60343 processed (1816.0s elapsed, ~1228s remaining)
40000/60343 processed (2016.7s elapsed, ~1026s remaining)
44000/60343 processed (2211.3s elapsed, ~821s remaining)
48000/60343 processed (2406.9s elapsed, ~619s remaining)
52000/60343 processed (2626.2s elapsed, ~421s remaining)
56000/60343 processed (2836.6s elapsed, ~220s remaining)
60000/60343 processed (3052.6s elapsed, ~17s remaining)


,date,headline,ticker,score
0,2020-06-10 07:33:26,Tech Stocks And FAANGS Strong Again To Start D...,AAPL,0.986816
1,2020-06-10 04:14:08,10 Biggest Price Target Changes For Wednesday,AAPL,0.033020
2,2020-06-10 03:53:47,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL,0.578125
3,2020-06-10 03:19:25,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL,0.997070
4,2020-06-10 02:27:11,Apple To Let Users Trade In Their Mac Computer...,AAPL,0.001511


### Save -- download this from Kaggle's Output tab and bring it back locally

In [13]:
out_name = "scored_SAMPLE.csv" if SAMPLE_MODE else "scored.csv"
df_news.to_csv(out_name, index=False)
print("Saved", df_news.shape[0], "scored headlines to", out_name)
if SAMPLE_MODE:
    print("This was a SAMPLE_MODE smoke test only -- set SAMPLE_MODE = False in the config "
          "cell near the top and use Kernel > Restart & Run All for the real full run.")
df_news["score"].describe()


Saved 60343 scored headlines to scored.csv


count    60343.000000
mean         0.202742
std          0.523677
min         -1.000000
25%          0.002563
50%          0.092285
75%          0.656738
max          1.000000
Name: score, dtype: float64